# Metrics for object detection & segmentation

_Metrics & Evaluation — measuring models, data & statistics_

**Score boxes, masks, and tracks by how much they overlap the truth.**

Every metric here starts from one idea: how much does the prediction overlap the truth?

       For a box or a mask, measure the area they share, then divide by the area they cover together. That ratio is IoU (Intersection over Union), also called the Jaccard index.

---

This notebook is a practice scaffold for the **Metrics for object detection & segmentation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — torchmetrics

In [ ]:
import numpy as np

# ---- from-scratch IoU for two boxes (x0, y0, x1, y1) ----
def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

print("box IoU:", round(box_iou((1, 0, 7, 8), (2, 1, 8, 9)), 3))  # ~0.574

# ---- from-scratch IoU + Dice for two binary masks ----
def mask_iou(p, g):
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    return inter / union if union else 0.0

def dice(p, g):                      # Dice == F1 for masks
    inter = np.logical_and(p, g).sum()
    denom = p.sum() + g.sum()
    return 2 * inter / denom if denom else 0.0

pred = np.array([[1, 1, 0], [1, 0, 0]], dtype=bool)
gt   = np.array([[1, 1, 1], [0, 0, 0]], dtype=bool)
print("mask IoU:", mask_iou(pred, gt), "Dice:", round(dice(pred, gt), 3))  # 0.5, 0.667

# ---- real detection mAP with torchmetrics (COCO mAP@[.5:.95]) ----
import torch
from torchmetrics.detection import MeanAveragePrecision

preds = [dict(
    boxes=torch.tensor([[10., 10., 50., 50.]]),
    scores=torch.tensor([0.9]),
    labels=torch.tensor([0]),
)]
target = [dict(
    boxes=torch.tensor([[12., 12., 52., 52.]]),
    labels=torch.tensor([0]),
)]
metric = MeanAveragePrecision(iou_type="bbox")   # also "segm" for masks
metric.update(preds, target)
res = metric.compute()
print("COCO mAP@[.5:.95]:", res["map"].item())
print("mAP@.50:", res["map_50"].item(), " mAP@.75:", res["map_75"].item())

# ---- real segmentation metrics with torchmetrics ----
from torchmetrics.segmentation import MeanIoU, DiceScore
miou = MeanIoU(num_classes=2)        # mIoU over classes (background + object)
ds = DiceScore(num_classes=2)        # Dice / F1 for masks
pred_t = torch.tensor([[[1, 1, 0], [1, 0, 0]]])
gt_t   = torch.tensor([[[1, 1, 1], [0, 0, 0]]])
print("mIoU:", miou(pred_t, gt_t).item())
print("Dice:", ds(pred_t, gt_t).item())
# Panoptic Quality (PQ) lives in torchmetrics.detection.PanopticQuality.

## Visualize the data & results

_Take one real handwritten 0 from load_digits, box its ink, and score candidate boxes by IoU — how fast does overlap fall as a box shifts or loosens?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

# one real 8x8 handwritten "0"
img = load_digits().images[0]
ink = img > 4                       # ink pixels
ys, xs = np.where(ink)
# true box from the ink extent, as (x0, y0, x1, y1)
true_box = (xs.min(), ys.min(), xs.max() + 1, ys.max() + 1)   # (1, 0, 7, 8)

def iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union else 0.0

x0, y0, x1, y1 = true_box
candidates = {
    "perfect":         true_box,
    "loose +1 margin": (x0-1, y0-1, x1+1, y1+1),
    "shift +1px":      (x0+1, y0+1, x1+1, y1+1),
    "too small":       (x0+1, y0+1, x1-1, y1-1),
    "big shift +3px":  (x0+3, y0+2, x1+3, y1+2),
}
for name, box in candidates.items():
    print(name, round(iou(true_box, box), 3))
# perfect 1.0 | loose +1 margin 0.6 | shift +1px 0.574 | too small 0.5 | big shift +3px 0.231

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A predicted box and the true box each cover $48$ square pixels and overlap in $35$. What is their IoU?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Union $= |A| + |B| - |A\cap B| = 48 + 48 - 35 = 61$. — _Subtract the shared area once so it is not double counted._
- IoU $= 35 / 61$. — _Overlap divided by union._

**Answer:** IoU $= 35/61 \approx 0.574$. With a $\tau = 0.5$ threshold it counts as a hit; with $\tau = 0.75$ it would be a miss.

</details>

**Problem 2.** Two masks each cover $3$ pixels and share $2$. Compute IoU and Dice, and say which is larger.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- IoU $= 2 / (3 + 3 - 2) = 2/4$. — _Shared pixels over the union of pixels._
- Dice $= 2\times 2 / (3 + 3) = 4/6$. — _Twice the shared pixels over the total of both masks._

**Answer:** IoU $= 0.5$, Dice $\approx 0.667$. Dice is larger — it always exceeds IoU for the same masks, so never compare the two numbers directly.

</details>

**Problem 3.** A detector scores IoU $\ge 0.5$ on every box but barely overlaps when the cutoff rises to $0.75$. Which single metric exposes this, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pick a metric that averages over IoU cutoffs. — _A single $\tau = 0.5$ threshold treats a sloppy box and a perfect box the same._
- Report COCO mAP@[.5:.95]. — _Averaging AP over $\tau = 0.5,0.55,\ldots,0.95$ rewards tight localization and drops on loose boxes._

**Answer:** COCO mAP@[.5:.95]. It averages mAP across ten IoU thresholds, so loose-but-passing boxes can no longer hide behind a single lenient cutoff.

</details>